# Player を継承して自作 AI を作る

Player.choose_command() をオーバーライドする。

In [ ]:
%pip install -q jpoke

In [ ]:
from __future__ import annotations

from jpoke import Battle, Player
from jpoke.enums import Command

In [ ]:
class CustomePlayer(Player):
    def choose_selection(self, battle: Battle) -> list[int]:
        """
        Player.choose_selection() は、チームのポケモンのインデックスを選出順に並べたリストを返す。
        ここでは素早さの高い順に選出する。
        """
        indexes = list(range(len(self.team)))
        speeds = [self.team[i].stats["spe"] for i in indexes]
        order = sorted(indexes, key=lambda i: speeds[i], reverse=True)
        return order[:battle.n_selected]
        
    def choose_command(self, battle: Battle) -> Command:
        """
        Player.choose_command() は、行動を表す Command オブジェクトを返す。
        ここでは、ダメージが最大になる行動を選択する。
        """
        commands = battle.get_available_commands(self)
        return max(commands, key=lambda command: self._damage(battle, command))
    
    def _damage(self, battle: Battle, command: Command) -> int:
        if not command.is_move:
            return -1

        move = battle.command_to_move(self, command) 
        opponent = battle.opponent(self)   
        damages = battle.calc_damages(
            attacker=self.active_pokemon,
            defender=opponent.active_pokemon,
            move=move,
            critical=move.guaranteed_crit # 確定急所を考慮する
        )
        return damages[0] # 0: 最低ダメージ

In [ ]:
# TODO: 2vs2の対戦にする
player1 = CustomePlayer("StrongestMovePlayer")
player1.add_pokemon("ピカチュウ", move_names=["でんこうせっか", "かみなり", "なきごえ"])

player2 = Player("Player 2")
player2.add_pokemon("フシギダネ", move_names=["たいあたり"])

In [ ]:
battle = Battle(player1, player2, seed=1)
battle.play_out(max_turns=100)
winner = battle.winner
print(f"勝者: {winner.username if winner else '引き分け（ターン上限）'}")
battle.print_logs("all")
print("-" * 50)  # print_logs() の出力とその後のprint()を視覚的に区切る

試してみよう: move_power() に「相手のタイプ相性が悪い技は除外する」といった
条件を加えると、より賢い方策に発展させられる